In [1]:
import os

# Uremote_edite you config directory as needed
os.environ["PSYNC_CONFIG_DIR"] = "../../../config"

# Collections

The Plex service implements the standard collection interfaces that are common across all services.

## Library Collection

The {py:class}`PlexLibrarySectionCollection <plistsync.services.plex.PlexLibrarySectionCollection>` represents content specific to one of your Plex Libraries. (Plex uses different naming: in the frontend they refer to them as "Libraries", in the backend they are "Sections".)

In [2]:
from plistsync.services.plex import PlexLibrarySectionCollection

library = PlexLibrarySectionCollection(section_name_or_id="Music")

### Iterating Tracks

The Plex Library collection implements the {py:class}`TrackStream <plistsync.core.collection.TrackStream>` protocol, so you can simply iterate over its tracks.

In [3]:
# Iterate over tracks
for count, track in enumerate(library.tracks):
    print(f"{track.id} -> {track.title} by {track.artists}")
    if count == 4:
        break

106387 -> If I Could by ['Break, Jack Boston, Kyo']
106388 -> Betamax by ['Break, Total Science']
106389 -> Inside by ['Break, SpectraSoul']
106390 -> Ain't No Turnin' Back by ['Break']
106391 -> Free Your Mind - Break Remix by ['Break, Singing Fats']


In [4]:
# Preload metadata for all tracks in your library, to speed things up later
# ~ 6sec for 20k tracks
library.preload()

In [5]:
# should be noticably faster now
for count, track in enumerate(library.tracks):
    print(f"{track.id} -> {track.title} by {track.artists}")
    if count == 4:
        break

106387 -> If I Could by ['Break, Jack Boston, Kyo']
106388 -> Betamax by ['Break, Total Science']
106389 -> Inside by ['Break, SpectraSoul']
106390 -> Ain't No Turnin' Back by ['Break']
106391 -> Free Your Mind - Break Remix by ['Break, Singing Fats']


### Track Lookup

The Plex Library collection supports lookup by:
- local IDs (`plex_id` but you may also know it as the `rating_key`)
- global IDs, in particular `isrc`

Currently, lookups simply iterate over tracks. Speed them up by preloading.

In [6]:
# By plex id
if plex_track := library.find_by_local_ids({"plex_id": "106387"}):
    print(plex_track)

# silence type errors
assert plex_track is not None

Track[Break, Jack Boston, Kyo > If I Could, hash: -2824579694457712665]


### Track Lookup via ISRC

The lookup via isrc (global id) is more involved (as-of-now):

The problem is that Plex does not expose the isrc as metadata.
As a workaround, we read it from the tags contained in the file (which requires access to your server's data, and might be slow).

In the example below, we run plistsync on a mac that is not the plex server, and have mounted the library data to `/Volumes/music`. On the server, the data sits at `/media/music`, which is also the location Plex will return for any tracks.

This requires us to map the local to the remote path, for which we use a {py:class}`PathRewrite <plistsync.core.PathRewrite>`.

In [7]:
from pathlib import Path

from plistsync.core import PathRewrite

# known to plex
remote_path: Path = plex_track.path  # type: ignore
print(f"{remote_path=}")

# rewrite to my mac, and check it exists
path_rewrite = PathRewrite.from_str("/media/music/", "/Volumes/music/")
local_path = path_rewrite.apply(remote_path)
print(f"{local_path=}")
print(f"{local_path.is_file()=}")

remote_path=PosixPath('/media/music/clean/Various Artists/10 Years of Symmetry/01 If I Could [1047kbps].flac')
local_path=PosixPath('/Volumes/music/clean/Various Artists/10 Years of Symmetry/01 If I Could [1047kbps].flac')
local_path.is_file()=True


In [8]:
# isrc not available in plex track:
print(f"{plex_track=}")
print(f"isrc via plex -> {plex_track.isrc}")

# isrc not available in file:
local_track = plex_track.get_local_track(path_rewrite=path_rewrite)
print(f"{local_track=}")
print(f"isrc from file -> {local_track.isrc}")

plex_track=Track[Break, Jack Boston, Kyo > If I Could, hash: -2824579694457712665]
isrc via plex -> None
local_track=Track[Break, Jack Boston, Kyo > If I Could, hash: 5975837560554330968]
isrc from file -> GBXJH1000082


In [9]:
# Currently only feasable for small libraries, when connected via network.
library.find_by_global_ids({"isrc": "GBXJH1000082"}, path_rewrite=path_rewrite)

Track[Break, Jack Boston, Kyo > If I Could, hash: -2824579694457712665]

## Playlist Collection

The {py:class}`PlexPlaylistCollection <plistsync.services.plex.PlexPlaylistCollection>` represents a playlist of an authenticated user. 

### Retrieving playlists

You can retrieve all playlists of a user using the library's {py:meth}`PlexLibrarySectionCollection.playlists <plistsync.services.plex.PlexLibrarySectionCollection.playlists>` property.

In [10]:
for pl in library.playlists:
    print(f"{pl.id:7d} {pl.name} ({len(pl)} tracks)")

 110044 _42k (4 tracks)
 108489 _inbox (67 tracks)
 109614 _plistsync (5 tracks)
 108528 Abriss (13 tracks)
 106822 Bootlegish (34 tracks)
  73313 Bootlegs (141 tracks)
  75104 Bremen 24 (17 tracks)
 106960 Bungle after Dark (21 tracks)
 107611 Burg Beats 25 (69 tracks)
  49567 Chill Background (30 tracks)
  74050 Contrast Vibes (22 tracks)
 109370 Daily Routines (29 tracks)
  76714 Dark n Dirty DnB (27 tracks)
 108530 DnB Classics (7 tracks)
 108310 Dub (3 tracks)
 107612 Dubstep (7 tracks)
 107964 Dubstep into DnB (5 tracks)
 108956 FemPWR (7 tracks)
  72250 G-town 24 (23 tracks)
 109888 Goldi Sommerfest 25 (52 tracks)
  74048 Gorillaman (1 tracks)
  76645 Healing DnB (27 tracks)
 106821 Hous-Ey (13 tracks)
  72883 Jungle is massive (26 tracks)
  74049 Latino DnB (58 tracks)
 107745 Lenzman Vibes (46 tracks)
 108061 Liquid DnB (12 tracks)
 109369 M-Tech (62 tracks)
 109154 Major Happy (14 tracks)
  75196 mellow dnb (8 tracks)
 109153 Mellow Vocals DnB (36 tracks)
 111046 My New Playl

If you just want a specific playlist, you can use the library's {py:meth}`PlexLibraryCollection.get_playlist <plistsync.services.plex.PlexLibraryCollection.get_playlist>` method to retrieve a playlist.

In [11]:
# Can get via namer or id
if pl := library.get_playlist(
    name="DnB Classics"
    # id = 108530
):
    print(f"Found playlist: {pl.id} {pl.name} ({len(pl.tracks)} tracks)")
else:
    print("Playlist not found.")

Found playlist: 108530 DnB Classics (7 tracks)


:::{note}
This method supports lookup by various identifiers, currently ``name=`` and ``id=``.
Lookup by name will return ``None`` if no matching playlist is found, while lookups by other identifiers will raise a ``ValueError`` if the playlist cannot be resolved.
:::

### Creating playlists

Playlists are created in two steps:
1. Create a local instance
2. Link it to the server

This two-step pattern ~~allows~~ will soon allow us to work with playlists, add tracks and edit details, and only communicate with the server once, when we are done.

In [30]:
from plistsync.services.plex import PlexPlaylistCollection

pl = PlexPlaylistCollection(
    library=library,
    name="My New Playlist"
)

# currently, server does not know of this playlist.
pl.remote_associated

False

In [31]:
pl.remote_create()

pl.remote_associated

True

### Updating a playlist

For updating a playlist, you should use the playlist's {py:meth}`PlexPlaylistCollection.remote_edit <plistsync.services.plex.PlexPlaylistCollection.remote_edit>` context manager. This ensures that all changes are properly saved back to Spotify when you exit the context. This also minifies changes and therefore reduces API calls.


In [33]:
pl = library.get_playlist(name="My New Playlist")

assert pl is not None

with pl.remote_edit():
    pl.description = "Updated Description"

DEBUG:plistsync:Resolved playlist 111071 with title 'My New Playlist'.


You can add tracks to a playlist using the same context manager, again this will add the tracks when you exit the context.

In [35]:
from plistsync.services.plex import PlexTrack

# find some tracks to play around with
new_tracks : list[PlexTrack] = list(
    library.find_many_by_local_ids(
        [
            {"plex_id": "111017"},
            {"plex_id": "111018"},
            {"plex_id": "111023"},
            {"plex_id": "111024"},
        ]
    )
) # type: ignore # TODO: use generic type in ABC
new_tracks  = list(filter(None, new_tracks))
new_tracks

[Track[gyrofield > Fallen In Deep (Trail remix), hash: -366064468446176705],
 Track[Rizzle > Mirage (Thread remix), hash: 7873411444636672139],
 Track[Rueben & PHI NIX > In My Mind (Tom Finster remix), hash: 6596213650973632158],
 Track[Skantia > 2Drill (Calyx remix), hash: -5439711326211403300]]

In [36]:
assert pl is not None
with pl.remote_edit():
    pl.tracks = new_tracks

[t.id for t in pl.tracks]

['111017', '111018', '111023', '111024']

In [37]:
assert pl is not None
with pl.remote_edit():
    pl.tracks = [pl.tracks[-1]] + pl.tracks[0:3]

[t.id for t in pl.tracks]

HTTPError: 404 Client Error: Not Found for url: http://tiny.flicker-drum.ts.net:32400/playlists/111071/items/111024/move

To reorder tracks in a playlist, you can change the order of the {py:attr}`SpotifyPlaylistCollection.tracks <plistsync.services.spotify.SpotifyPlaylistCollection.tracks>` list within the context manager.

In [ ]:
with pl.edit():
    pl.tracks.insert(0, pl.tracks.pop())